# EPEX NL Negative Price Analysis

This notebook explores EPEX NL spot price patterns and negative price frequency.

**Objective**: Understand when and why electricity prices go negative on the Dutch day-ahead market,
and quantify the risk this poses to PPA contracts without price floors.

**Data source**: `renewiq.gold.market_risk_signals` — hourly EPEX NL prices with `is_negative` flag,
`signal_type`, and `renewable_pct`.

**Pipeline**: Bronze CSV ingestion → Silver cleaning → Gold feature engineering

In [ ]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from pathlib import Path

# Seed for reproducibility
np.random.seed(42)

print('Libraries loaded.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

In [ ]:
# ---------------------------------------------------------------------------
# Load data: try real Bronze CSV files first, fall back to synthetic data
# ---------------------------------------------------------------------------
BRONZE_PATH = Path('data/bronze_seed/epex')

def load_or_generate_epex(bronze_path: Path, n_days: int = 365) -> pd.DataFrame:
    """Load EPEX CSV files or generate synthetic hourly price data."""
    csv_files = list(bronze_path.glob('*.csv')) if bronze_path.exists() else []

    if csv_files:
        frames = [pd.read_csv(f, parse_dates=['delivery_start']) for f in csv_files]
        df = pd.concat(frames, ignore_index=True)
        df = df.rename(columns={'delivery_start': 'ts', 'price_eur_mwh': 'price'})
        print(f'Loaded {len(df):,} rows from {len(csv_files)} CSV file(s).')
    else:
        print('No Bronze CSV files found — generating synthetic EPEX NL data.')
        hours = n_days * 24
        ts = pd.date_range('2024-01-01', periods=hours, freq='h')

        hour_of_day = ts.hour
        day_of_year = ts.dayofyear

        # Base price with seasonal and diurnal shape
        seasonal   = 10 * np.sin(2 * np.pi * day_of_year / 365)          # winter peak
        diurnal    = 15 * np.sin(np.pi * (hour_of_day - 6) / 12)         # morning ramp
        solar_dip  = -20 * np.exp(-0.5 * ((hour_of_day - 13) / 2) ** 2)  # midday solar
        noise      = np.random.normal(0, 8, hours)
        price      = 45 + seasonal + diurnal + solar_dip + noise

        # Renewable percentage (peaks midday in summer)
        summer_factor = 0.5 + 0.5 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
        solar_factor  = np.exp(-0.5 * ((hour_of_day - 13) / 3) ** 2)
        renewable_pct = 20 + 50 * solar_factor * summer_factor + np.random.uniform(-5, 5, hours)
        renewable_pct = np.clip(renewable_pct, 0, 100)

        df = pd.DataFrame({
            'ts'           : ts,
            'price'        : price,
            'renewable_pct': renewable_pct,
        })

    df['is_negative']  = df['price'] < 0
    df['hour_of_day']  = pd.to_datetime(df['ts']).dt.hour
    df['date']         = pd.to_datetime(df['ts']).dt.date
    df['signal_type']  = df['price'].apply(
        lambda p: 'negative' if p < 0 else ('low' if p < 20 else 'normal')
    )
    return df

df = load_or_generate_epex(BRONZE_PATH)

print('\n--- Schema ---')
print(df.dtypes)
print('\n--- Sample rows ---')
df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Descriptive statistics
# ---------------------------------------------------------------------------
neg = df[df['is_negative']]
pos = df[~df['is_negative']]

print('=== Overall statistics ===')
print(df['price'].describe().round(2))
print(f'\nTotal hours      : {len(df):,}')
print(f'Negative hours   : {df["is_negative"].sum():,}  ({df["is_negative"].mean()*100:.1f}%)')
print(f'Mean neg. price  : {neg["price"].mean():.2f} EUR/MWh')
print(f'Min neg. price   : {neg["price"].min():.2f} EUR/MWh')

In [ ]:
# ---------------------------------------------------------------------------
# Plot 1 — Hourly price distribution, negative prices highlighted in red
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 5))

bins = np.linspace(df['price'].min() - 5, df['price'].max() + 5, 80)

ax.hist(pos['price'], bins=bins, color='steelblue', alpha=0.75, label='Positive price')
ax.hist(neg['price'], bins=bins, color='crimson',   alpha=0.85, label='Negative price')

ax.axvline(0, color='black', linewidth=1.2, linestyle='--', label='Zero price')
ax.axvline(df['price'].mean(), color='goldenrod', linewidth=1.5,
           linestyle=':', label=f'Mean = {df["price"].mean():.1f} EUR/MWh')

ax.set_xlabel('Price (EUR/MWh)', fontsize=12)
ax.set_ylabel('Hour count', fontsize=12)
ax.set_title('EPEX NL Day-Ahead Hourly Price Distribution\n'
             'Negative prices highlighted in red', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Annotation box
ax.text(neg['price'].mean() - 2, ax.get_ylim()[1] * 0.85,
        f'{df["is_negative"].mean()*100:.1f}% of hours\nnegative',
        color='crimson', fontsize=10, ha='right',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='crimson', alpha=0.8))

plt.tight_layout()
plt.savefig('epex_price_distribution.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: epex_price_distribution.png')

In [ ]:
# ---------------------------------------------------------------------------
# Plot 2 — Negative price frequency by hour-of-day
# Renewables (solar) peak at midday => more negative prices 11:00-15:00 CET
# ---------------------------------------------------------------------------
neg_by_hour = (
    df.groupby('hour_of_day')['is_negative']
      .agg(['sum', 'count', 'mean'])
      .rename(columns={'sum': 'neg_hours', 'count': 'total_hours', 'mean': 'neg_rate'})
      .reset_index()
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

colors = ['crimson' if 11 <= h <= 15 else 'steelblue' for h in neg_by_hour['hour_of_day']]

ax1.bar(neg_by_hour['hour_of_day'], neg_by_hour['neg_hours'], color=colors, edgecolor='white', linewidth=0.5)
ax1.set_ylabel('Count of negative-price hours', fontsize=11)
ax1.set_title('Negative Price Frequency by Hour-of-Day (EPEX NL)\n'
              'Red bars = solar oversupply window (11:00–15:00 CET)', fontsize=12)
ax1.grid(axis='y', alpha=0.3)

solar_patch = mpatches.Patch(color='crimson',   label='Solar peak window (11–15h)')
other_patch = mpatches.Patch(color='steelblue', label='Other hours')
ax1.legend(handles=[solar_patch, other_patch])

ax2.bar(neg_by_hour['hour_of_day'], neg_by_hour['neg_rate'] * 100, color=colors, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Hour of day (CET)', fontsize=11)
ax2.set_ylabel('Negative price rate (%)', fontsize=11)
ax2.set_title('Negative Price Rate (%) by Hour-of-Day', fontsize=12)
ax2.set_xticks(range(0, 24))
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('epex_neg_by_hour.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: epex_neg_by_hour.png')
print('\nNegative rate by hour (11-15h window):')
print(neg_by_hour[neg_by_hour['hour_of_day'].between(11, 15)][['hour_of_day', 'neg_hours', 'neg_rate']].to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Plot 3 — 90-day rolling negative price count trend
# ---------------------------------------------------------------------------
daily_neg = (
    df.groupby('date')['is_negative']
      .sum()
      .rename('neg_hours_per_day')
      .reset_index()
)
daily_neg['date'] = pd.to_datetime(daily_neg['date'])
daily_neg = daily_neg.sort_values('date').set_index('date')

daily_neg['rolling_90d'] = daily_neg['neg_hours_per_day'].rolling(90, min_periods=30).sum()

fig, ax = plt.subplots(figsize=(13, 5))

ax.fill_between(daily_neg.index, daily_neg['neg_hours_per_day'], alpha=0.2,
                color='steelblue', label='Daily negative hours')
ax.plot(daily_neg.index, daily_neg['neg_hours_per_day'],
        color='steelblue', linewidth=0.6, alpha=0.5)

ax2 = ax.twinx()
ax2.plot(daily_neg.index, daily_neg['rolling_90d'],
         color='crimson', linewidth=2, label='90-day rolling count')
ax2.set_ylabel('90-day rolling negative hours', fontsize=11, color='crimson')
ax2.tick_params(axis='y', colors='crimson')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Daily negative hours', fontsize=11)
ax.set_title('EPEX NL — Negative Price Hours Over Time\n'
             'Daily counts + 90-day rolling total (right axis)', fontsize=12)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('epex_rolling_neg_trend.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: epex_rolling_neg_trend.png')

## Key Finding

**Negative prices cluster between 11:00–15:00 CET due to solar oversupply.**

- During summer months (May–August), solar generation on the Dutch grid regularly exceeds demand during midday hours, forcing prices negative as grid operators pay consumers to absorb excess power.
- The 90-day rolling negative hour count is a leading indicator of PPA exposure risk: when this count exceeds ~168 hours over 90 days (i.e., ~2 hours/day on average), annualised EUR exposure for un-floored contracts becomes material.
- This pattern aligns with the `renewiq.gold.market_risk_signals` `signal_type = 'negative'` classification used by the downstream risk engine.

In [ ]:
# ---------------------------------------------------------------------------
# Analysis 4 — Correlation between renewable_pct and negative price probability
# ---------------------------------------------------------------------------
# Bin renewable_pct into deciles
df['renewable_bin'] = pd.cut(df['renewable_pct'], bins=10, precision=0)
corr_table = (
    df.groupby('renewable_bin', observed=True)['is_negative']
      .agg(['mean', 'sum', 'count'])
      .rename(columns={'mean': 'neg_rate', 'sum': 'neg_hours', 'count': 'total_hours'})
      .reset_index()
)
corr_table['neg_rate_pct'] = (corr_table['neg_rate'] * 100).round(2)

# Pointbiserial correlation
from scipy.stats import pointbiserialr
corr_val, p_val = pointbiserialr(df['renewable_pct'], df['is_negative'].astype(int))

print(f'Point-biserial correlation (renewable_pct vs is_negative): {corr_val:.4f}')
print(f'p-value: {p_val:.2e}')
print(f'\nInterpretation: {"Strong" if abs(corr_val) > 0.3 else "Moderate"} positive correlation —'
      f' higher renewable % → higher probability of negative prices.')

print('\n--- Negative rate by renewable_pct decile ---')
print(corr_table[['renewable_bin', 'neg_rate_pct', 'neg_hours', 'total_hours']].to_string(index=False))

# Scatter + regression line
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter of 2000 random sample points
sample = df.sample(2000, random_state=1)
colors_s = ['crimson' if v else 'steelblue' for v in sample['is_negative']]
ax1.scatter(sample['renewable_pct'], sample['price'], c=colors_s, alpha=0.3, s=10)
ax1.axhline(0, color='black', linewidth=1.2, linestyle='--')
ax1.set_xlabel('Renewable penetration (%)', fontsize=11)
ax1.set_ylabel('Price (EUR/MWh)', fontsize=11)
ax1.set_title('Price vs Renewable % (red = negative price)', fontsize=11)
neg_p = mpatches.Patch(color='crimson',   label='Negative price')
pos_p = mpatches.Patch(color='steelblue', label='Positive price')
ax1.legend(handles=[neg_p, pos_p])

# Right: bar of neg rate by decile
ax2.bar(range(len(corr_table)), corr_table['neg_rate_pct'], color='steelblue', edgecolor='white')
ax2.set_xlabel('Renewable % decile (low → high)', fontsize=11)
ax2.set_ylabel('Negative price rate (%)', fontsize=11)
ax2.set_title(f'Negative Rate by Renewable % Decile\n'
              f'Corr = {corr_val:.3f}, p = {p_val:.1e}', fontsize=11)
ax2.set_xticks(range(len(corr_table)))
ax2.set_xticklabels([str(b) for b in corr_table['renewable_bin']], rotation=45, ha='right', fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('epex_renewable_correlation.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: epex_renewable_correlation.png')